In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder.appName("Week5-Kaggle").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [3]:
df = spark.read.csv("/ecommerce_transactions.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Rows:", df.count(), "| Columns:", len(df.columns))

root
 |-- Transaction_ID: integer (nullable = true)
 |-- User_Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Purchase_Amount: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)

+--------------+---------------+---+-------+----------------+---------------+--------------+----------------+
|Transaction_ID|      User_Name|Age|Country|Product_Category|Purchase_Amount|Payment_Method|Transaction_Date|
+--------------+---------------+---+-------+----------------+---------------+--------------+----------------+
|             1|       Ava Hall| 63| Mexico|        Clothing|         780.69|    Debit Card|      2023-04-14|
|             2|    Sophia Hall| 59|  India|          Beauty|         738.56|        PayPal|      2023-07-30|
|             3|Elijah Thompson| 26| France|           Books|         178.34|   Credit Card|  

In [4]:
COL_ID        = "User_Name"          # closest thing to a customer identifier
COL_DATE      = "Transaction_Date"
COL_COUNTRY   = "Country"            # substitute for "region"
COL_CATEGORY  = "Product_Category"
COL_AMOUNT    = "Purchase_Amount"    # substitute for "sales"
COL_AGE       = "Age"
COL_PAYMENT   = "Payment_Method"     # substitute for "subscription"

***Q1. Key limitations of traditional MapReduce that make Spark a preferred choice?***
Traditional MapReduce writes intermediate results to disk after every Map and Reduce phase, which makes multi-step jobs slow due to constant disk I/O. It also handles iterative algorithms poorly since it re-reads data from disk on every pass, forces every computation into a rigid two-stage Map-Shuffle-Reduce model, doesn't support interactive or ad-hoc querying since jobs are batch-only and slow to start, and requires a lot of boilerplate code for even simple operations. Spark solves these by keeping data in memory across operations, supporting flexible chained transformations, and offering an interactive, much less verbose API.

***Q2. How Spark's In-Memory Computing speeds up iterative ML algorithms?***
In disk-based systems, every iteration of an algorithm reads the dataset from disk, processes it, and writes results back — so training something like a 100-epoch model means 100 full read/write cycles from the slowest part of the system. Spark instead loads the dataset into memory once (using .cache() or .persist()), so every following iteration reads directly from RAM instead of disk. This is why Spark's ML algorithms like logistic regression or k-means are much faster than a MapReduce-based equivalent — the expensive disk I/O cost is paid once instead of once per iteration.

In [5]:
before = df.count()
df_dedup = df.dropDuplicates([COL_ID, COL_DATE])
print("Duplicates removed:", before - df_dedup.count())

Duplicates removed: 13846


In [6]:
df.select(COL_COUNTRY).distinct().show()   # run this first to see valid country names

+---------+
|  Country|
+---------+
|  Germany|
|   France|
|    India|
|      USA|
|   Mexico|
|       UK|
|   Canada|
|   Brazil|
|    Japan|
|Australia|
+---------+



In [7]:
df.filter(F.col(COL_COUNTRY) == "USA") \
  .groupBy(COL_CATEGORY) \
  .agg(F.round(F.avg(COL_AMOUNT), 2).alias("avg_amount")) \
  .orderBy(F.desc("avg_amount")) \
  .show()

+----------------+----------+
|Product_Category|avg_amount|
+----------------+----------+
|          Beauty|    524.26|
|  Home & Kitchen|    523.33|
|     Electronics|    513.63|
|          Sports|    510.71|
|           Books|    506.79|
|         Grocery|    505.89|
|        Clothing|    503.37|
|            Toys|    496.93|
+----------------+----------+



***Q5. Difference between .na.drop() and .na.fill()?***
.na.drop() removes entire rows containing null values, so the record is lost. .na.fill() replaces nulls with a specified default instead, keeping the row. This dataset has no status column, so Payment_Method is used as a substitute example for the fill demonstration.

In [8]:
from pyspark.sql.functions import col, sum as spark_sum
df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------------+---------+---+-------+----------------+---------------+--------------+----------------+
|Transaction_ID|User_Name|Age|Country|Product_Category|Purchase_Amount|Payment_Method|Transaction_Date|
+--------------+---------+---+-------+----------------+---------------+--------------+----------------+
|             0|        0|  0|      0|               0|              0|             0|               0|
+--------------+---------+---+-------+----------------+---------------+--------------+----------------+



In [9]:
print("Nulls before:", df.filter(F.col(COL_PAYMENT).isNull()).count())
df_filled = df.na.fill({COL_PAYMENT: "Unknown"})
print("Nulls after:", df_filled.filter(F.col(COL_PAYMENT).isNull()).count())

Nulls before: 0
Nulls after: 0


In [10]:
df.groupBy(COL_COUNTRY).count().filter(F.col("count") > 100).orderBy(F.desc("count")).show()

+---------+-----+
|  Country|count|
+---------+-----+
|   Canada| 5082|
|   Mexico| 5059|
|  Germany| 5047|
|    India| 4996|
|   France| 4993|
|Australia| 4985|
|      USA| 4979|
|    Japan| 4960|
|       UK| 4951|
|   Brazil| 4948|
+---------+-----+



***Q7. How immutability of Spark DataFrames affects data cleaning steps***
Spark DataFrames are immutable, so no transformation like .drop() or .filter() changes the original DataFrame — each one returns a new DataFrame instead. This means you have to reassign the result (df = df.filter(...)) or chain multiple steps together, otherwise the cleaned version gets lost. It also makes pipelines safer and more reproducible, since each step is a well-defined transformation and you can always go back to the original data if something breaks. The tradeoff is you sometimes need extra variable names to track each stage of cleaning, since you can't update one DataFrame directly.

In [11]:
df.select(COL_PAYMENT).distinct().show()

+----------------+
|  Payment_Method|
+----------------+
|     Credit Card|
|     Net Banking|
|          PayPal|
|      Debit Card|
|Cash on Delivery|
|             UPI|
+----------------+



In [12]:
df.filter((F.col(COL_AGE).between(18, 30)) & (F.col(COL_PAYMENT) == "Credit Card")).show()

+--------------+-----------------+---+-------+----------------+---------------+--------------+----------------+
|Transaction_ID|        User_Name|Age|Country|Product_Category|Purchase_Amount|Payment_Method|Transaction_Date|
+--------------+-----------------+---+-------+----------------+---------------+--------------+----------------+
|             3|  Elijah Thompson| 26| France|           Books|         178.34|   Credit Card|      2023-09-17|
|             7|     Oliver Clark| 27|Germany|  Home & Kitchen|         341.73|   Credit Card|      2024-03-13|
|            37|        Noah Hall| 21| France|            Toys|          474.1|   Credit Card|      2024-10-06|
|            49|  Elijah Anderson| 20|    USA|          Sports|         189.64|   Credit Card|      2023-08-03|
|            66|   James Thompson| 18|Germany|        Clothing|         269.38|   Credit Card|      2024-02-28|
|            84|Isabella Anderson| 21|  India|     Electronics|         999.57|   Credit Card|      2024

***Q9. Why handle null values before aggregations like sum() or avg()?***
Spark's sum() and avg() ignore nulls by default instead of treating them as 0, so a column with many nulls can silently produce an average calculated over fewer rows than expected, giving a misleadingly clean-looking number. Also, avg() divides by the count of non-null values, not the total row count, so comparing averages across groups with different null rates isn't a fair comparison. If a null-heavy column feeds into further calculations, the null can also propagate and break downstream results. Handling nulls first makes it clear whether a missing value should be treated as 0, excluded, or imputed, so the aggregation's meaning is well-defined instead of ambiguous.

***Q12. Remove rows with null identifier OR empty region value***
This dataset has no email/username columns, so User_Name (null check) and Country (empty-string check) are used as substitutes to demonstrate the same OR-condition null-cleaning logic.

In [13]:
before = df.count()
df_clean = df.filter(~(F.col(COL_ID).isNull() | (F.col(COL_COUNTRY) == "")))
after = df_clean.count()
print(f"Before: {before} | After: {after} | Removed: {before - after}")

Before: 50000 | After: 50000 | Removed: 0


In [14]:
df.agg(
    F.min(COL_AMOUNT).alias("min_amt"),
    F.max(COL_AMOUNT).alias("max_amt"),
    F.round(F.mean(COL_AMOUNT), 2).alias("mean_amt")
).show()

+-------+-------+--------+
|min_amt|max_amt|mean_amt|
+-------+-------+--------+
|   5.04| 999.98|  503.16|
+-------+-------+--------+



In [15]:
final = (
    df.dropDuplicates([COL_ID, COL_DATE])
      .na.fill({COL_AMOUNT: 0})
      .groupBy(COL_CATEGORY)
      .agg(F.round(F.sum(COL_AMOUNT), 2).alias("total_revenue"))
      .orderBy(F.desc("total_revenue"))
)
final.show()

+----------------+-------------+
|Product_Category|total_revenue|
+----------------+-------------+
|           Books|   2322397.87|
|          Sports|   2313870.83|
|         Grocery|   2306608.18|
|        Clothing|   2302847.25|
|            Toys|   2275884.43|
|     Electronics|   2252446.98|
|  Home & Kitchen|    2245878.6|
|          Beauty|    2214909.1|
+----------------+-------------+



In [16]:
df.select("Transaction_Date").show(5, truncate=False)

+----------------+
|Transaction_Date|
+----------------+
|2023-04-14      |
|2023-07-30      |
|2023-09-17      |
|2023-06-21      |
|2024-10-29      |
+----------------+
only showing top 5 rows


***Q11. The "Shuffle" process during grouping, and why it's a wide transformation?***
A shuffle is the process of redistributing data across partitions so all rows with the same key end up together. During a groupBy(), Spark needs every row with the same group value (like the same category or city) to be on the same partition before it can compute an aggregate for that group, but those rows are usually scattered across many partitions after loading. The shuffle involves writing data out bucketed by the grouping key, moving that data across partitions (often over the network), and then computing the aggregate once all matching rows are together. This is called a wide transformation because the output partitions depend on many or all input partitions, unlike a narrow transformation like filter() where each output partition depends on just one input partition — this is why shuffles are usually the most expensive part of a Spark job.


In [17]:
df_ts = df.withColumn("event_time", F.to_timestamp("Transaction_Date")).drop("Transaction_Date")
df_ts.select("event_time").show(5)
print("Nulls after cast:", df_ts.filter(F.col("event_time").isNull()).count())

+-------------------+
|         event_time|
+-------------------+
|2023-04-14 00:00:00|
|2023-07-30 00:00:00|
|2023-09-17 00:00:00|
|2023-06-21 00:00:00|
|2024-10-29 00:00:00|
+-------------------+
only showing top 5 rows
Nulls after cast: 0


***Q14. Risk of using inferSchema=True with messy/inconsistent date formats?***
inferSchema=True makes Spark sample the data and guess a data type per column, but for dates this is risky because Spark picks one pattern based on the sample it sees. If the real column mixes formats, rows that don't match get silently turned into nulls instead of raising an error. Sometimes Spark may not recognize the column as a date type at all and leave it as plain text. Since inference only scans a sample rather than the whole dataset, a rare format might never get seen during inference and then fails later. The safer approach is to read date columns as plain text and parse them explicitly with to_timestamp using known format strings, instead of trusting automatic inference.

***Insights***

Deduplication removed 13,846 duplicate rows out of 50,000 based on User_Name + Transaction_Date — a large chunk, likely because User_Name repeats across many transactions in this dataset rather than being a unique customer ID.
Payment_Method, User_Name, and Transaction_Date had zero nulls in this dataset — cleaner than a typical real-world dataset, so the null-handling logic (Q5, Q9, Q12) is demonstrated correctly even though it found nothing to clean here; the same code would handle nulls if they existed.
Books had the highest total revenue (~2.32M) among all product categories, closely followed by Sports and Grocery.
Transaction_Date parsed into event_time with zero failed casts, since this dataset uses one consistent date format — unlike a genuinely messy dataset, where Q14's warning about blind inferSchema use would actually cause silent data loss.